# GoalLab Stage 14: Distill search and deliver the final briefing

Created by Shivam Bharadwaj · [Course home](../../../README.md) · [Stage map](../STAGES.md)

[Stage 13](13_budgeted_reasoning_search.ipynb) · [Integrated demo](../README.md#run-the-reference-solution)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bshivambharadwaj/Reinforcement-Learning-Course/blob/main/projects/goallab/apply-theory/14_learning_from_search.ipynb)

**Build:** fit a separate student from accepted training-workspace searches, evaluate it on held-out evidence, and save a final GoalLab briefing.

**Run:** use the full repository, install `requirements.txt` (Stage 09 also uses `requirements-modern.txt`), then restart the kernel and run all cells. Every stage reconstructs its inputs from shared GoalLab modules; earlier notebook execution is not required. The project progression is cumulative in capabilities, without hidden notebook state.

Code: MIT. Text and plots: CC BY 4.0. Results below are reproduced from the supplied GoalLab cases; the evaluation section explains their scope and failure cases.

**Learn the algorithm first:** [Course Notebook 14](../../../notebooks/14_learning_from_search.ipynb). Then use this GoalLab stage to apply it to evidence gathering and briefing verification.


In [1]:
import sys, subprocess, platform, tempfile, json
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'rl_course').is_dir()), Path.cwd())
if IN_COLAB and not (ROOT / 'rl_course').exists():
    ROOT = Path('/content/Reinforcement-Learning-Course')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/bshivambharadwaj/Reinforcement-Learning-Course.git', str(ROOT)], check=True)
if not (ROOT / 'rl_course' / 'goallab.py').exists():
    raise FileNotFoundError('Use the complete repository revision containing the GoalLab stages.')
sys.path.insert(0, str(ROOT))
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Markdown, display
from rl_course import goallab as goal
from rl_course import goallab_learning as learning
torch.set_num_threads(2)
%matplotlib inline
INK, GOLD, SAGE, PLUM, PAPER = '#292332', '#af824a', '#768165', '#87708c', '#f5f1e8'
plt.rcParams.update({'figure.figsize': (9, 4), 'figure.dpi': 110, 'figure.facecolor': PAPER,
    'axes.facecolor': PAPER, 'text.color': INK, 'axes.labelcolor': INK,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=[GOLD, SAGE, PLUM, INK]), 'axes.grid': True, 'grid.alpha': .15})
def table(headers, rows):
    display(Markdown('| ' + ' | '.join(headers) + ' |\n|' + '|'.join(['---'] * len(headers)) + '|\n' +
        '\n'.join('| ' + ' | '.join(str(x).replace('|', '&#124;').replace('\n', '<br>') for x in row) + ' |' for row in rows)))
def learning_plot(runs):
    for label, run in runs.items():
        h = run['history']
        plt.plot(h[:, 0], h[:, 1], label=label)
    plt.xlabel('Training environment actions'); plt.ylabel('Exact policy return')
    plt.legend(); plt.tight_layout(); plt.show()
def report_policies(runs):
    table(['Controller', 'Exact return', 'Held-out briefing success', 'Calls/case'],
          [[label, round(run['env'].values(run['policy'])[0][run['env'].start_index], 3),
            f"{learning.heldout(run['policy'])['success']:.1%}", learning.heldout(run['policy'])['calls']]
           for label, run in runs.items()])
workspace = goal.make_workspace(7, 'test')
print(f'{goal.VERSION} | Python {platform.python_version()} | NumPy {np.__version__} | Torch {torch.__version__} | CPU')

GoalLab-v1 | Python 3.12.4 | NumPy 1.26.4 | Torch 2.11.0+cu130 | CPU


## Shared contract

One source is an approved measurement for the requested period. Other sources are drafts or forecasts, deliberately newer and numerically different. Training rows use 0..3; held-out rows use 4..7 and a different period. Authority rules stay the same. `search` reveals IDs and timestamps; `read` reveals content. A valid briefing needs the correct value, period, and citation.

The simulator and models are readable teaching code, not a security boundary. Final `verify()` is outside the policy interface. The first eight stages use a finite two-source abstraction: read twice, choose a citation, then submit or claim completion. Claiming completion produces no briefing. This fixed horizon makes exact comparisons possible; later stages vary investigation budgets.

## Learning objectives and the stage contract

Fit a separate student from training traces; account for rejected-trace costs; test the learned policy under a changed correlation.

**Input:** the shared evidence workspace or the preceding component reconstructed below. **Output:** the capability named in this stage, evaluated against the common briefing contract. Read the worked calculation before running the full experiment; then inspect the implementation and predict the constraint exercise.

## Work it out: acceptance changes the training distribution

Suppose 20 accepted trajectories all choose the older source. With one pseudocount per binary outcome, the fitted probability of choosing the newest source is `1/22`, not exactly zero. The counts describe selected training traces, not every possible workspace.

If authority always correlates with age in training, this compact student can learn age rather than the evidence rule. The timestamp-shift exercise is therefore an essential part of the result, not an optional warning.

In [2]:
accepted=20
p_newest=(0+1)/(accepted+2)
print('Smoothed probability of newest source:',p_newest)
assert np.isclose(p_newest,1/22)
print('The final evaluator must test changed correlations independently of the fitting set.')

Smoothed probability of newest source: 0.045454545454545456
The final evaluator must test changed correlations independently of the fitting set.


## Build the component

The central implementation is included here so you can step through and edit the update or tool logic. The shared modules retain the same reference implementation for the integrated project. Inspect shapes, terminal handling, frozen quantities, and the evaluator boundary before training.

In [3]:
from rl_course import goallab_search as search
FrozenProposer=search.FrozenProposer
make_workspace=goal.make_workspace
verify=goal.verify
def distill(cases=160, budget=24, seed=7, acceptance='verified'):
    """Fit a new categorical student from accepted TRAINING-workspace traces.

    With no accepted examples, keep the teacher rather than inventing evidence.
    This is counts-based filtered imitation, not PPO/GRPO or LLM fine-tuning.
    """
    if acceptance not in {'verified', 'proxy'}:
        raise ValueError(acceptance)
    teacher = FrozenProposer()
    traces, cost = [], 0
    for i in range(cases):
        workspace = make_workspace(i, 'train')
        run = search.search(workspace, budget=budget, seed=seed * 100003 + i, policy=teacher)
        cost += run['spent']
        accept = verify(workspace, run['artifact'])['success'] if acceptance == 'verified' else run['artifact'] is not None
        if accept:
            newest = max(range(2), key=lambda j: workspace.sources[j].updated)
            source, offset, citation = run['prefix']
            traces.append((source == newest, offset == 0, citation == source))
    probabilities = (np.sum(traces, axis=0) + 1) / (len(traces) + 2) if traces else None
    student = FrozenProposer(*probabilities.tolist()) if traces else teacher
    return {'teacher': teacher, 'student': student, 'accepted': len(traces), 'teacher_units': cost}

## 1. Search first, learn later

The teacher stays frozen during each investigation. Afterward, collect accepted training trajectories and estimate three new categorical probabilities with one pseudocount per binary outcome. This is filtered imitation, not policy-gradient RL or an LLM distillation claim.

The student learns a recency shortcut under this controlled two-source generator, whose approved source is always older. That shortcut must be tested under changed timestamp rules before any generalization claim.

In [4]:
from rl_course import goallab_search as search
def summarize(rows):
    groups={}
    for r in rows: groups.setdefault((r['method'],r['budget']),[]).append(r)
    table(['Method','Budget','Verified success','Oracle opportunity','Spent','Completion'],
          [[m,b,*[round(np.mean([r[k] for r in rs]),3) for k in ['success','oracle','spent','completion']]] for (m,b),rs in groups.items()])
def search_plot(rows):
    for method in dict.fromkeys(r['method'] for r in rows):
        group=[r for r in rows if r['method']==method]
        budgets=sorted({r['budget'] for r in group})
        plt.plot(budgets,[np.mean([r['success'] for r in group if r['budget']==b]) for b in budgets],marker='o',label=method)
    plt.xlabel('Maximum search work units'); plt.ylabel('Verified briefing success'); plt.legend(fontsize=8); plt.tight_layout(); plt.show()

trained=distill(160,24,7)
print('Accepted training traces:',trained['accepted'],'Teacher work units:',trained['teacher_units'])
print('Teacher:',trained['teacher']); print('Student:',trained['student'])
rows=[]
for label,policy,method,budget in [('Teacher single',trained['teacher'],'single',3),('Teacher search',trained['teacher'],'best_of_n',24),('Student single',trained['student'],'single',3)]:
    current=search.benchmark(cases=100,budgets=(budget,),methods=(method,),policy=policy)
    for r in current:r['method']=label
    rows.extend(current)
summarize(rows)

Accepted training traces: 86 Teacher work units: 3840
Teacher: FrozenProposer(newest_probability=0.75, exact_probability=0.65, matching_citation_probability=0.8)
Student: FrozenProposer(newest_probability=0.011363636363636364, exact_probability=0.9886363636363636, matching_citation_probability=0.9886363636363636)


| Method | Budget | Verified success | Oracle opportunity | Spent | Completion |
|---|---|---|---|---|---|
| Teacher single | 3 | 0.13 | 0.13 | 3.0 | 1.0 |
| Teacher search | 24 | 0.557 | 0.557 | 24.0 | 1.0 |
| Student single | 3 | 0.977 | 0.977 | 3.0 | 1.0 |

## 2. Submit the selected artifact and persist a review record

The final candidate still goes through the gateway. Search work and committed tool calls are distinct costs: report both. Never repair an unsuccessful candidate using the final evaluator before measuring it.

In [5]:
result=search.search(workspace,'single',3,7,policy=trained['student'])
session=result['session']
if session is not None:
    submission=session.execute('submit',result['artifact'])
    with tempfile.TemporaryDirectory(prefix='goallab-final-') as output:
        record=goal.save_record(session,Path(output)/'briefing.json')
        print(json.dumps(record,indent=2))
print('Search work units:',result['spent'])
print('Independent final result:',goal.verify(workspace,result['artifact']))

{
  "version": "GoalLab-v1",
  "case": "test-7",
  "budget": 4,
  "events": [
    {
      "action": "read",
      "argument": 1,
      "result": {
        "ok": true,
        "source": {
          "source_id": "test-7-source-1",
          "status": "approved",
          "kind": "measurement",
          "period": "2026-02",
          "updated": 1,
          "rows": [
            6,
            5
          ]
        }
      }
    },
    {
      "action": "submit",
      "argument": {
        "value": 11,
        "citation": "test-7-source-1",
        "period": "2026-02",
        "abstained": false
      },
      "result": {
        "ok": true,
        "submitted": true
      }
    }
  ],
  "artifact": {
    "value": 11,
    "citation": "test-7-source-1",
    "period": "2026-02",
    "abstained": false
  },
  "evaluation": {
    "success": true,
    "supported": true,
    "value_correct": true
  }
}
Search work units: 3
Independent final result: {'success': True, 'supported': True, 'value

## 3. Account for the cost of fitting

Teacher-search data generation costs work even when traces are rejected. A simple work-unit break-even divides that cost by per-request search savings. It does not include model-training FLOPs, and it is meaningful only when the student meets the required quality threshold.

In [6]:
saving=24-3
print('Data-generation-only break-even requests:',int(np.ceil(trained['teacher_units']/saving)))
print('Check quality first; cost savings do not compensate automatically for unsupported briefings.')

Data-generation-only break-even requests: 183
Check quality first; cost savings do not compensate automatically for unsupported briefings.


## Interpret the evidence

The fitted student can score highly on the original test generator and fail after timestamp reversal. The improvement is real within the original distribution and brittle outside it. The reference solution includes that failed stress test as part of the deliverable.

## Change a constraint: predict first

Reverse which document looks newest while preserving authority and content. Does the student still identify valid evidence?

### Worked solution

Run this after writing your prediction. Compare the changed condition with the reference condition above.

In [7]:
from dataclasses import replace
shifted=[]
for i in range(100):
    w=goal.make_workspace(i,'test')
    w=replace(w,sources=tuple(replace(s,updated=9 if goal.usable(s,w.period) else 1) for s in w.sources))
    r=search.search(w,'single',3,1001+i,policy=trained['student'])
    shifted.append(goal.verify(w,r['artifact'])['success'])
print('Student success after timestamp shift:',np.mean(shifted))
print('A strong original score can conceal a shortcut. Re-evaluate the evidence baseline and search before deploying under this shift.')

Student success after timestamp shift: 0.0
A strong original score can conceal a shortcut. Re-evaluate the evidence baseline and search before deploying under this shift.


## What this stage contributes

The final stage integrates GoalLab evidence, candidate search, a fitted student, gateway submission, and independent evaluation. The timestamp-shift experiment exposes a limitation rather than hiding it. Theory: [learning from search](../../../chapters/47-learning-from-search/README.md), [evaluation](../../../chapters/48-evaluating-inference-time-reasoning/README.md).

[Stage 13](13_budgeted_reasoning_search.ipynb) · [Integrated demo](../README.md#run-the-reference-solution)